In [147]:
import numpy as np
from sklearn.linear_model import LogisticRegression
import pandas as pd

path = r"C:\Users\oskar\OneDrive\Desktop\4 Semester\Dataproject\Federated-dental-risk-vol2\federated-dental-risk-prediction\data\raw\synthetic_dataset_A_non-iid.csv"
data = pd.read_csv(path)


# DISCLAIMER
This document tries to implement a softmax model. But when we were working on this we realized that it was not actually very useful for our project. Our primary goal is to produce an ordinal ranking of datapoints according to risk scores. For this purpose, a linear model is sufficient, since we are mainly interested in relative ordering rather than calibrated class probabilities.

Furthermore, the softmax model predicts probabilities over mutually exclusive classes whose probabilities must sum to one. The outputs are normalized probabilities over mutually exclusive classes, so increasing the probability of one class necessarily decreases the probabilities of the others. This makes the model not suitable for interpreting each class score as an independent risk measure.

Because of this we are not using the this for anything in the project. But we wanted to document the work and considerations regarding it. 

In [148]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def prepare_data(df, begin=1, end=27, cli=False):

    df = df.copy()

    # Split først (undgå leakage)
    train_df, test_df = train_test_split(df, test_size=0.25, random_state=16)

    # Kolonner der skal skaleres
    features = df.columns.tolist()[begin:end]

    scaler = StandardScaler()

    # Fit kun på train
    train_df[features] = scaler.fit_transform(train_df[features])
    test_df[features] = scaler.transform(test_df[features])

    # Tilføj intercept hvis ønsket
    train_df.insert(0, "intercept", 1.0)
    test_df.insert(0, "intercept", 1.0)

    # Client håndtering (valgfri – hvis du vil flytte den frem)
    if cli:
        # Flyt Client kolonne frem (hvis den findes)
        cols = ["Client"] + [col for col in train_df.columns if col != "Client"]
        train_df = train_df[cols]
        test_df = test_df[cols]

    return train_df, test_df

train, test = prepare_data(data)



In [149]:
#Under 0.1% af datapunkter har mere end 1 komplication.
(train[[
    "Risk_AlveolarOsteitis",
    "Risk_SecondaryInfection",
    "Risk_NerveDysesthesia",
    "Risk_Bleeding"
]].sum(axis=1) > 1).mean()

np.float64(0.009617021276595744)

In [150]:
# Vi vil forsøge at lave softmax med 5 output klasser. Vi antager dermed at man maksimalt kan få 1 komplication. 
# Der er dermed 5 output klasser hvor den ene er ingen komplication. 
# Bemærk jeg har valgt AlveolarOsteitis > Bleeding > NerveDysesthesia > SecondaryInfection
def create_softmax_target(df):
    y = np.zeros(len(df), dtype=int)

    y[df["Risk_SecondaryInfection"] == 1] = 1
    y[df["Risk_NerveDysesthesia"] == 1] = 2
    y[df["Risk_Bleeding"] == 1] = 3
    y[df["Risk_AlveolarOsteitis"] == 1] = 4

    return y

def features(df):
    targets = create_softmax_target(df)
    variabels = df.columns[1:27].tolist()
    n = len(df)
    d = len(variabels)
    X = np.zeros((n,d))
    for i in range(d):
        X[:,i] = df[variabels[i]]
    return X, targets, variabels

def Softmax_centralized(train):
    X, targets, variabels = features(train)
    model = LogisticRegression(
    multi_class="multinomial",   # <- vigtig (softmax)
    solver="lbfgs",             # understøtter multinomial
    max_iter=1000
    )
    model.fit(X, targets)

    return model, variabels

model, vari = Softmax_centralized(train)
test_vari, a,b = features(test)
probs = model.predict_proba(test_vari)  # softmax outputs (5 klasser)
preds_alt = np.where(probs[:, 0] < 0.8, probs[:, 1:].argmax(axis=1) + 1, 0)


c:\Users\oskar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [151]:
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

y_test = create_softmax_target(test)

f1_macro = f1_score(y_test, preds_alt, average="macro")
f1_weighted = f1_score(y_test, preds_alt, average="weighted")
acc = accuracy_score(y_test, preds_alt)

print("Accuracy:", acc)
print("F1 macro:", f1_macro)
print("F1 weighted:", f1_weighted)
print(classification_report(y_test, preds_alt))


Accuracy: 0.7080851063829787
F1 macro: 0.23471412004111752
F1 weighted: 0.73973775922957
              precision    recall  f1-score   support

           0       0.89      0.78      0.83     10102
           1       0.15      0.10      0.12       494
           2       0.00      0.00      0.00        76
           3       0.00      0.00      0.00        65
           4       0.15      0.39      0.22      1013

    accuracy                           0.71     11750
   macro avg       0.24      0.25      0.23     11750
weighted avg       0.79      0.71      0.74     11750



c:\Users\oskar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\oskar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\oskar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In a softmax (multiclass) setup, the model assigns a single probability distribution over mutually exclusive classes, meaning all class probabilities must sum to one. This makes the probabilities relative rather than independent, so they cannot be interpreted as absolute risks for each complication. As a result, it is not meaningful to define low, mid, and high risk levels for each complication individually. Such categorization requires independent probability estimates, which are not provided by a softmax model. 
I continue with the federated setup. The code is abit messy since LogisticRegression only makes softmax if we have multible classes, some clients only have 2 classes.

In [129]:
from sklearn.linear_model import LogisticRegression
from scipy.special import softmax
import numpy as np

def federated_softmax(df, weighted=True):
    client_ids = sorted(df["Client"].unique())
    local_models = []

    for client_id in client_ids:
        client_df = df[df["Client"] == client_id]
        model, vari = Softmax_centralized(client_df)

        d = len(vari)
        coef_full = np.zeros((5, d))
        intercept_full = np.zeros(5)

        classes = model.classes_

        # Hvis client kun har én klasse, giver den ingen mening at bruge
        if len(classes) < 2:
            continue

        # Binær case: sklearn gemmer kun én række i coef_
        if len(classes) == 2 and model.coef_.shape[0] == 1:
            cls0, cls1 = classes[0], classes[1]

            coef_full[cls0, :] = -model.coef_[0, :]
            coef_full[cls1, :] =  model.coef_[0, :]

            intercept_full[cls0] = -model.intercept_[0]
            intercept_full[cls1] =  model.intercept_[0]

        # Multiclass case
        else:
            for local_idx, cls in enumerate(classes):
                coef_full[cls, :] = model.coef_[local_idx, :]
                intercept_full[cls] = model.intercept_[local_idx]

        n = len(client_df)
        local_models.append((coef_full, intercept_full, n, vari))

    if len(local_models) == 0:
        raise ValueError("Ingen clients kunne bruges.")

    if weighted:
        total_n = sum(n for _, _, n, _ in local_models)
        global_coef = sum(coef * n for coef, _, n, _ in local_models) / total_n
        global_intercept = sum(intercept * n for _, intercept, n, _ in local_models) / total_n
    else:
        global_coef = np.mean([coef for coef, _, _, _ in local_models], axis=0)
        global_intercept = np.mean([intercept for _, intercept, _, _ in local_models], axis=0)

    variables = local_models[0][3]
    return global_coef, global_intercept, variables


def predict_federated_softmax(test, global_coef, global_intercept, variables):
    X = test[variables].to_numpy()
    logits = X @ global_coef.T + global_intercept
    probs = softmax(logits, axis=1)
    preds = np.argmax(probs, axis=1)
    return probs, preds


In [130]:
global_coef, global_intercept, variables = federated_softmax(train, weighted=True)

probs, preds = predict_federated_softmax(
    test,
    global_coef,
    global_intercept,
    variables
)



c:\Users\oskar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\oskar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\oskar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\oskar\AppData\Local\Programs\Python\P

Federated metrics:

In [143]:
preds_08 = np.where(probs[:, 0] < 0.8, probs[:, 1:].argmax(axis=1) + 1, 0)
y_test = create_softmax_target(test)

print("F1 macro:", f1_score(y_test, preds_08, average="macro"))
print(classification_report(y_test, preds_08))

F1 macro: 0.22277113892918005
              precision    recall  f1-score   support

           0       0.87      0.73      0.80     10102
           1       0.10      0.12      0.11       494
           2       0.05      0.05      0.05        76
           3       0.00      0.00      0.00        65
           4       0.11      0.29      0.16      1013

    accuracy                           0.66     11750
   macro avg       0.23      0.24      0.22     11750
weighted avg       0.77      0.66      0.70     11750



Centralized metrics:

In [146]:
print("F1 macro:", f1_score(y_test, preds_alt, average="macro"))
print(classification_report(y_test, preds_alt))

F1 macro: 0.23471412004111752
              precision    recall  f1-score   support

           0       0.89      0.78      0.83     10102
           1       0.15      0.10      0.12       494
           2       0.00      0.00      0.00        76
           3       0.00      0.00      0.00        65
           4       0.15      0.39      0.22      1013

    accuracy                           0.71     11750
   macro avg       0.24      0.25      0.23     11750
weighted avg       0.79      0.71      0.74     11750



c:\Users\oskar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\oskar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\oskar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave